In [ ]:
import sys
import pickle as pk
from pathlib import Path as ph

In [ ]:
# Add parent directory to sys.path
sys.path.append(str(ph().resolve().parent))
from src.gpt import GPT
from src.functions.runtime import from_file, towa_file, get_directory_files

In [ ]:
runtime_path_inp = input("Enter the runtime path ('same', '<path>'): ").strip().lower()
runtime_uuid_inp = input("Enter the model configuration ('<uuid>', 'all'): ").lower()

In [ ]:
########################
# Runtime variables
########################

if runtime_path_inp == "same":
    runtime_path = "."
else:
    runtime_path = runtime_path_inp

runtime_uuid = runtime_uuid_inp

config_path = f"{runtime_path}/config"
inputs_path = f"{runtime_path}/inputs"
outputs_path = f"{runtime_path}/outputs"
models_path = f"{runtime_path}/models"

In [ ]:
########################
# Extract model components
########################

def process_extract_model_components(folder_path, file_prefix, runtime_uuid):
    runtime_type_inp = input("Enter the extract type ('from_pkl_obj', 'from_pkl_wgt'): ").lower()
    runtime_type = runtime_type_inp
    runtime_uuids = []
    if runtime_uuid == "all":
        # If a all is provided, add ithem all to the list
        runtime_uuids = get_directory_files(folder_path, file_prefix)
    else:
        # If a specific UUID is provided, add it to the list
        runtime_uuids.append(runtime_uuid)

    # Loop through each GUID to extract weights
    for runtime_uuid in runtime_uuids:
        if runtime_type == "from_pkl_obj":
            # Load the model from a single file
            models_path_inp = f"{models_path}/model_{runtime_uuid}.pkl"
            model_ = from_file(models_path_inp, "binary")

            # Extract all weight arrays from the model
            weights_dict = {
                # Token and position embeddings
                'wte_weight': model_.wte.weight,
                'wpe_weight': model_.wpe.weight,
                
                # Final layer norm
                'ln_f_gamma': model_.ln_f.gamma,
                'ln_f_beta': model_.ln_f.beta,
                
                # Language model head
                'lm_head_weight': model_.lm_head.weight,
                'lm_head_bias': model_.lm_head.bias if model_.lm_head.bias is not None else None,
            }

            # Add block weights
            for i, block in enumerate(model_.blocks):
                # Layer norms
                weights_dict[f'block_{i}_ln1_gamma'] = block.ln_1.gamma
                weights_dict[f'block_{i}_ln1_beta'] = block.ln_1.beta
                weights_dict[f'block_{i}_ln2_gamma'] = block.ln_2.gamma
                weights_dict[f'block_{i}_ln2_beta'] = block.ln_2.beta
                
                # Multi-head attention
                weights_dict[f'block_{i}_mha_q_weight'] = block.mha.q_proj.weight
                weights_dict[f'block_{i}_mha_k_weight'] = block.mha.k_proj.weight
                weights_dict[f'block_{i}_mha_v_weight'] = block.mha.v_proj.weight
                weights_dict[f'block_{i}_mha_c_weight'] = block.mha.c_proj.weight
                weights_dict[f'block_{i}_mha_c_bias'] = block.mha.c_proj.bias if block.mha.c_proj.bias is not None else None
                
                # MLP
                weights_dict[f'block_{i}_mlp_fc_weight'] = block.mlp.c_fc.weight
                weights_dict[f'block_{i}_mlp_fc_bias'] = block.mlp.c_fc.bias if block.mlp.c_fc.bias is not None else None
                weights_dict[f'block_{i}_mlp_proj_weight'] = block.mlp.c_proj.weight
                weights_dict[f'block_{i}_mlp_proj_bias'] = block.mlp.c_proj.bias if block.mlp.c_proj.bias is not None else None

            # Store the model weights to a single file
            models_path_out = f"{models_path}/model_{runtime_uuid}.weights"
            towa_file(models_path_out, "json")

        elif runtime_type == "from_pkl_wgt":
            models_path_inp = f"{models_path}/model_{runtime_uuid}.pkl"
            weights_dict = from_file(models_path_inp, "binary")

            # Convert numpy arrays to lists for JSON serialization
            json_weights_dict = {}
            for key, value in weights_dict.items():
                if value is not None and hasattr(value, 'tolist'):
                    json_weights_dict[key] = value.tolist()
                else:
                    json_weights_dict[key] = value

            # Store the model weights to a single file
            models_path_out = f"{models_path}/model_{runtime_uuid}.weights"
            towa_file(models_path_out, "json")


In [ ]:
def process_extract_model_information(folder_path, file_prefix, runtime_uuid):
    runtime_uuids = []
    if runtime_uuid == "all":
        # If a all is provided, add ithem all to the list
        runtime_uuids = get_directory_files(folder_path, file_prefix)
    else:
        # If a specific UUID is provided, add it to the list
        runtime_uuids.append(runtime_uuid)

    models_params = []
    # Loop through each GUID to extract weights
    for runtime_uuid in runtime_uuids:
        file_name = f"{file_prefix}_{runtime_uuid}.json"
        file_path = f"{folder_path}/{file_name}"

        config = from_file(file_path, "json")

        # Instantiate the model
        model = GPT(config)
        # Load the pre-trained or fine-tuned model parameters
        model_path = f"{models_path}/model_{runtime_uuid}.json"
        model_json = from_file(model_path, "json")
        model.params_from_dict(model_json)
        # Count the number of parameters in the model
        params = model.count_parameters()
        model_params = { "model": runtime_uuid, "params": params }
        models_params.append(model_params)
    
    print(f"Number of parameters in the model: {models_params}")

In [ ]:
def process_add_model_information(folder_path, file_prefix, runtime_uuid):
    runtime_uuids = []
    if runtime_uuid == "all":
        # If a all is provided, add ithem all to the list
        runtime_uuids = get_directory_files(folder_path, file_prefix)
    else:
        # If a specific UUID is provided, add it to the list
        runtime_uuids.append(runtime_uuid)

    models_params = []
    # Loop through each GUID to extract weights
    for runtime_uuid in runtime_uuids:
        file_name = f"{file_prefix}_{runtime_uuid}.json"
        file_path = f"{folder_path}/{file_name}"

        config = from_file(file_path, "json")
       
        # Instantiate the model
        model = GPT(config)
        # Load the pre-trained or fine-tuned model parameters
        model_path = f"{models_path}/model_{runtime_uuid}.json"
        model_json = from_file(model_path, "json")
        model.params_from_dict(model_json)
        # Count the number of parameters in the model
        params = model.count_parameters()
        config["runtime"]["model_params"] = params

        towa_file(file_path, "json", config)

In [ ]:
# === Run the script ===
folder_path = f"{runtime_path}/configs"
file_prefix = f"config"
_ = get_directory_files(folder_path, file_prefix)
#process_add_model_information(folder_path, file_prefix, runtime_uuid)